# Brain Ontology Integration Benchmark

This notebook demonstrates Boomer-Py's capabilities for integrating complex neuroanatomical ontologies across species and terminologies.

In [ ]:
import sys
sys.path.append('..')

from boomer.model import (
    KB, PFact, EquivalentTo, ProperSubClassOf, 
    MemberOfDisjointGroup, DisjointWith, SearchConfig
)
from boomer.search import solve
from boomer.renderers.markdown_renderer import MarkdownRenderer
from boomer.evaluator import evaluate_facts
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display, Markdown
import time

## 1. Create the Brain Knowledge Base

We'll create a KB that includes:
- Human brain anatomy (FMA)
- Mouse brain anatomy (Allen Brain Atlas)
- Clinical terminology (SNOMED CT)
- Historical neuroanatomical terms

In [ ]:
# Import the brain KB creation function
from brain_ontology_benchmark import create_brain_kb, create_extended_brain_kb

# Create the knowledge base
kb = create_brain_kb()

print(f"Knowledge Base Statistics:")
print(f"  Deterministic facts: {len(kb.facts)}")
print(f"  Probabilistic facts: {len(kb.pfacts)}")
print(f"  Entities with labels: {len(kb.labels)}")
print(f"  Search space: 2^{len(kb.pfacts)} = {2**len(kb.pfacts):,} possible combinations")

## 2. Visualize the Ontology Structure

In [ ]:
# Build a graph from the deterministic facts
G = nx.DiGraph()

# Add nodes and edges from facts
for fact in kb.facts:
    if isinstance(fact, ProperSubClassOf):
        G.add_edge(fact.sub, fact.sup, relation="subclass")
    elif isinstance(fact, DisjointWith):
        G.add_edge(fact.sub, fact.sibling, relation="disjoint")

# Identify different ontology sources by prefix
human_nodes = [n for n in G.nodes() if n.startswith('human_')]
mouse_nodes = [n for n in G.nodes() if n.startswith('mouse_')]
clinical_nodes = [n for n in G.nodes() if 'structure' in n]

print(f"Graph Statistics:")
print(f"  Total nodes: {G.number_of_nodes()}")
print(f"  Total edges: {G.number_of_edges()}")
print(f"  Human brain nodes: {len(human_nodes)}")
print(f"  Mouse brain nodes: {len(mouse_nodes)}")
print(f"  Clinical term nodes: {len(clinical_nodes)}")

# Show connected components
components = list(nx.weakly_connected_components(G))
print(f"\nConnected components: {len(components)}")
for i, comp in enumerate(components[:5]):
    print(f"  Component {i+1}: {len(comp)} nodes")

## 3. Analyze Probabilistic Mappings

In [ ]:
# Analyze probability distribution of mappings
probs = [pf.prob for pf in kb.pfacts]

# Create probability distribution plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(probs, bins=20, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Probability')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Mapping Probabilities')
axes[0].grid(True, alpha=0.3)

# Categorize by confidence level
high_conf = sum(1 for p in probs if p >= 0.8)
med_conf = sum(1 for p in probs if 0.5 <= p < 0.8)
low_conf = sum(1 for p in probs if p < 0.5)

# Pie chart
axes[1].pie([high_conf, med_conf, low_conf], 
           labels=['High (≥0.8)', 'Medium (0.5-0.8)', 'Low (<0.5)'],
           autopct='%1.1f%%',
           colors=['green', 'yellow', 'red'],
           alpha=0.7)
axes[1].set_title('Confidence Level Distribution')

plt.tight_layout()
plt.show()

# Show some example mappings by confidence level
print("\nExample High-Confidence Mappings:")
for pf in sorted(kb.pfacts, key=lambda x: x.prob, reverse=True)[:3]:
    print(f"  {pf.fact} (p={pf.prob:.2f})")

print("\nExample Low-Confidence Mappings:")
for pf in sorted(kb.pfacts, key=lambda x: x.prob)[:3]:
    print(f"  {pf.fact} (p={pf.prob:.2f})")

## 4. Configure and Run the Solver

In [ ]:
# Configure search parameters
config = SearchConfig(
    max_iterations=100000,
    max_candidate_solutions=1000,
    timeout_seconds=30,
    partition_initial_threshold=40,
    max_pfacts_per_clique=25,
)

print("Search Configuration:")
print(f"  Max iterations: {config.max_iterations:,}")
print(f"  Max candidate solutions: {config.max_candidate_solutions:,}")
print(f"  Timeout: {config.timeout_seconds}s")
print(f"  Partition threshold: {config.partition_initial_threshold}")
print(f"  Max pfacts per clique: {config.max_pfacts_per_clique}")

# Run the solver
print("\nRunning solver...")
start_time = time.time()
solution = solve(kb, config)
end_time = time.time()

print(f"\nSolver completed in {end_time - start_time:.2f} seconds")
print(f"Combinations explored: {solution.number_of_combinations:,}")
print(f"Satisfiable combinations: {solution.number_of_satisfiable_combinations:,}")
print(f"Proportion explored: {solution.proportion_of_combinations_explored:.2%}")
print(f"Solution confidence: {solution.confidence:.2%}")

## 5. Analyze Solution

In [ ]:
# Categorize results
accepted = []
rejected = []
uncertain = []

for spfact in solution.solved_pfacts:
    fact_data = {
        'fact': str(spfact.pfact.fact),
        'prior': spfact.pfact.prob,
        'posterior': spfact.posterior_prob,
        'accepted': spfact.truth_value
    }
    
    if spfact.truth_value and spfact.posterior_prob > 0.8:
        accepted.append(fact_data)
    elif not spfact.truth_value and spfact.posterior_prob < 0.2:
        rejected.append(fact_data)
    else:
        uncertain.append(fact_data)

print(f"Solution Summary:")
print(f"  Accepted mappings (high confidence): {len(accepted)}")
print(f"  Rejected mappings (low confidence): {len(rejected)}")
print(f"  Uncertain mappings: {len(uncertain)}")

# Create DataFrame for accepted mappings
if accepted:
    df_accepted = pd.DataFrame(accepted)
    df_accepted = df_accepted.sort_values('posterior', ascending=False)
    print("\nTop Accepted Mappings:")
    display(df_accepted.head(10)[['fact', 'prior', 'posterior']])

# Create DataFrame for rejected mappings
if rejected:
    df_rejected = pd.DataFrame(rejected)
    df_rejected = df_rejected.sort_values('prior', ascending=False)
    print("\nTop Rejected Mappings (that had high prior probability):")
    display(df_rejected.head(5)[['fact', 'prior', 'posterior']])

## 6. Cross-Species Mapping Analysis

In [ ]:
# Analyze cross-species mappings specifically
cross_species = []

for spfact in solution.solved_pfacts:
    fact = spfact.pfact.fact
    if isinstance(fact, EquivalentTo):
        # Check if it's a cross-species mapping
        if (('human' in fact.sub and 'mouse' in fact.equivalent) or 
            ('mouse' in fact.sub and 'human' in fact.equivalent)):
            cross_species.append({
                'human_term': fact.sub if 'human' in fact.sub else fact.equivalent,
                'mouse_term': fact.equivalent if 'mouse' in fact.equivalent else fact.sub,
                'prior_prob': spfact.pfact.prob,
                'posterior_prob': spfact.posterior_prob,
                'accepted': spfact.truth_value
            })

if cross_species:
    df_cross = pd.DataFrame(cross_species)
    df_cross_accepted = df_cross[df_cross['accepted'] == True].sort_values('posterior_prob', ascending=False)
    
    print("Cross-Species Mappings Analysis")
    print(f"Total cross-species mappings evaluated: {len(cross_species)}")
    print(f"Accepted: {sum(df_cross['accepted'])}")
    print(f"Rejected: {sum(~df_cross['accepted'])}")
    
    print("\nAccepted Cross-Species Mappings:")
    display(df_cross_accepted[['human_term', 'mouse_term', 'prior_prob', 'posterior_prob']])

## 7. Evaluate Against Ground Truth

In [ ]:
# Define ground truth based on expert consensus
ground_truth = [
    EquivalentTo(sub="human_cerebral_cortex", equivalent="mouse_isocortex"),
    EquivalentTo(sub="human_hippocampus", equivalent="mouse_hippocampal_formation"),
    EquivalentTo(sub="human_amygdala", equivalent="mouse_amygdalar_nuclei"),
    EquivalentTo(sub="human_frontal_lobe", equivalent="frontal_lobe_structure"),
    EquivalentTo(sub="neocortex", equivalent="human_cerebral_cortex"),
]

# Extract accepted facts from solution
accepted_facts = [
    spfact.pfact.fact
    for spfact in solution.solved_pfacts
    if spfact.truth_value and spfact.posterior_prob > 0.5
]

# Evaluate
eval_stats = evaluate_facts(ground_truth, accepted_facts)

print("Evaluation Metrics:")
print(f"  Precision: {eval_stats.precision:.2%}")
print(f"  Recall: {eval_stats.recall:.2%}")
print(f"  F1 Score: {eval_stats.f1:.2%}")
print(f"\n  True Positives: {eval_stats.tp}")
print(f"  False Positives: {eval_stats.fp}")
print(f"  False Negatives: {eval_stats.fn}")

if eval_stats.tp_list:
    print("\nCorrectly Identified Mappings:")
    for fact in eval_stats.tp_list:
        print(f"  ✓ {fact}")

if eval_stats.fn_list:
    print("\nMissed Mappings:")
    for fact in eval_stats.fn_list:
        print(f"  ✗ {fact}")

## 8. Scalability Analysis

In [ ]:
# Test scalability with different KB sizes
sizes = [10, 15, 20, 25, 30]
results = []

for size in sizes:
    # Create subset KB
    subset_kb = KB(
        facts=kb.facts,
        pfacts=kb.pfacts[:size],
        labels=kb.labels,
        name=f"Brain KB (size {size})"
    )
    
    config_scaled = SearchConfig(
        max_iterations=50000,
        timeout_seconds=20,
        partition_initial_threshold=15,
        max_pfacts_per_clique=10,
    )
    
    start = time.time()
    sol = solve(subset_kb, config_scaled)
    elapsed = time.time() - start
    
    results.append({
        'size': size,
        'time': elapsed,
        'combinations': sol.number_of_combinations,
        'confidence': sol.confidence,
        'search_space': 2**size
    })
    
    print(f"Size {size}: {elapsed:.2f}s, explored {sol.number_of_combinations:,}/{2**size:,}")

# Plot scalability results
df_scale = pd.DataFrame(results)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Time vs size
axes[0].plot(df_scale['size'], df_scale['time'], 'b-o')
axes[0].set_xlabel('KB Size (# pfacts)')
axes[0].set_ylabel('Time (seconds)')
axes[0].set_title('Execution Time Scaling')
axes[0].grid(True, alpha=0.3)

# Combinations explored vs size
axes[1].semilogy(df_scale['size'], df_scale['combinations'], 'g-o', label='Explored')
axes[1].semilogy(df_scale['size'], df_scale['search_space'], 'r--', label='Total Space')
axes[1].set_xlabel('KB Size (# pfacts)')
axes[1].set_ylabel('Combinations (log scale)')
axes[1].set_title('Search Space Exploration')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Confidence vs size
axes[2].plot(df_scale['size'], df_scale['confidence'], 'r-o')
axes[2].set_xlabel('KB Size (# pfacts)')
axes[2].set_ylabel('Solution Confidence')
axes[2].set_title('Solution Quality')
axes[2].set_ylim([0, 1])
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

display(df_scale)

## 9. Generate Report

In [ ]:
# Generate markdown report
renderer = MarkdownRenderer()
report = renderer.render(solution)

# Display first part of report
display(Markdown(report[:2000] + "\n\n...(truncated for display)"))

# Save full report
with open('brain_benchmark_solution.md', 'w') as f:
    f.write("# Brain Ontology Integration Solution\n\n")
    f.write(f"## Summary Statistics\n\n")
    f.write(f"- **Confidence**: {solution.confidence:.2%}\n")
    f.write(f"- **Combinations Explored**: {solution.number_of_combinations:,}\n")
    f.write(f"- **Time Elapsed**: {solution.time_elapsed:.2f}s\n")
    f.write(f"- **Accepted Mappings**: {len(accepted)}\n")
    f.write(f"- **Rejected Mappings**: {len(rejected)}\n\n")
    f.write(report)

print("\nFull report saved to 'brain_benchmark_solution.md'")

## 10. Extended Analysis with Larger KB

In [ ]:
# Try with extended KB for stress testing
extended_kb = create_extended_brain_kb()

print(f"Extended KB Statistics:")
print(f"  Probabilistic facts: {len(extended_kb.pfacts)}")
print(f"  Search space: 2^{len(extended_kb.pfacts)} ≈ {2**len(extended_kb.pfacts):.2e} combinations")

# Configure for larger KB
extended_config = SearchConfig(
    max_iterations=50000,
    max_candidate_solutions=100,
    timeout_seconds=60,
    partition_initial_threshold=30,
    max_pfacts_per_clique=20,
)

print("\nSolving extended KB (this may take a minute)...")
start = time.time()
extended_solution = solve(extended_kb, extended_config)
elapsed = time.time() - start

print(f"\nExtended KB Results:")
print(f"  Time: {elapsed:.2f}s")
print(f"  Partitions: {extended_solution.number_of_components}")
print(f"  Confidence: {extended_solution.confidence:.2%}")
print(f"  Explored: {extended_solution.proportion_of_combinations_explored:.4%} of search space")